# Telco Customer Churn — Case Study (Classification)

**Objective:** Predict whether a customer will churn (binary classification).

**Dataset:** IBM Telco Customer Churn (downloaded automatically on first run).

**Experimental protocol (mandatory):**
1. Train/test split **once** at the beginning.
2. Use **cross-validation only on the training set** for model comparison and selection.
3. Evaluate the **baseline (majority class)** first.
4. Compare advanced models against the baseline using the same protocol.
5. Retrain the chosen model on the full training set; evaluate **exactly once** on the held-out test set.
6. Report and justify metric choice (class imbalance => Accuracy vs Precision/Recall/F1/ROC-AUC).

In [ ]:
# Imports and configuration
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score    
import matplotlib.pyplot as plt

from src.config import RANDOM_STATE
from src.data_loading import load_telco
from src.preprocessing import make_classification_preprocessor
from src.splitting import split_classification, make_cv_classification
from src.metrics import cls_metrics
from src.evaluation import evaluate_classification_cv


# Reproducibility
np.random.seed(RANDOM_STATE)

## 1. Load data

In [ ]:
df = load_telco()
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Class distribution (focus: class imbalance)
df["Churn"].value_counts(normalize=True)

## 2. Exploratory Data Analysis (EDA)

**Look at the data first** — before any preprocessing, splitting, or training. EDA helps you understand distributions, spot missing values or data types issues, and form hypotheses about which features might relate to churn.

In [ ]:
# Missing values and data types
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotalCharges dtype:", df["TotalCharges"].dtype)
# TotalCharges is often read as object (e.g. empty strings); coerce to numeric to spot issues
tc_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges: non-numeric or missing count:", tc_numeric.isna().sum())

In [ ]:
# Target: Churn distribution (class imbalance matters for metric choice)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().plot(kind="bar", ax=ax[0], color=["steelblue", "coral"])
ax[0].set_title("Churn (counts)")
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)
df["Churn"].value_counts(normalize=True).plot(kind="bar", ax=ax[1], color=["steelblue", "coral"])
ax[1].set_title("Churn (proportions)")
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric features: summary and relation to Churn
print("Numeric columns — describe:")
print(df[["tenure", "MonthlyCharges"]].describe())
# TotalCharges: use numeric version for EDA (dropna for the plot)
df_eda = df.copy()
df_eda["TotalCharges_num"] = pd.to_numeric(df_eda["TotalCharges"], errors="coerce")
df_eda_numeric = df_eda[["tenure", "MonthlyCharges", "TotalCharges_num"]].dropna()

In [ ]:
# Numeric features by Churn (boxplots)
df_eda_numeric["Churn"] = df_eda.loc[df_eda_numeric.index, "Churn"].values
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for col, ax in zip(["tenure", "MonthlyCharges", "TotalCharges_num"], axes):
    df_eda_numeric.boxplot(column=col, by="Churn", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Churn")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature vs Churn (example: Contract)
pd.crosstab(df["Contract"], df["Churn"], normalize="index").round(3)

## 3. Define target and features, prepare data

- **Target:** `Churn` (binary: Yes → 1, No → 0). We care about identifying churners (positive class).
- **Features:** All columns except `customerID` (identifier) and `Churn` (target).
- **TotalCharges** is sometimes stored as text or has missing values; we coerce to numeric and impute if needed.

In [ ]:
# Coerce TotalCharges to numeric (empty strings become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# Drop rows with missing target or key numeric feature (optional: impute instead)
df_clean = df.dropna(subset=["TotalCharges", "Churn"]).copy()
# Encode target: Yes -> 1, No -> 0
df_clean["Churn"] = (df_clean["Churn"] == "Yes").astype(int)
print("Rows after dropna:", len(df_clean))

## Categorical features 

What if you have categorical features with **non-numerical** and **non-binary** values?

To showcase, we add a new (synthetic) column depicting the region of the customers. This is a categorical feature and has 4 values: North, South, East, West. 

In [ ]:
# (Optional) Synthetic categorical for teaching one-hot encoding: Region has no natural order.
df_clean["Region"] = np.random.RandomState(RANDOM_STATE).choice(
    ["North", "South", "East", "West"], size=len(df_clean)
)

df_clean.head()

In [ ]:

# Feature matrix X and target y (drop identifier and target)
drop_cols = ["customerID", "Churn"]
X = df_clean.drop(columns=drop_cols)
y = df_clean["Churn"]
print("X shape:", X.shape)
print("y value_counts:\n", y.value_counts())

## 4. Train/test split (once)

We split **once** and do not use the test set until the final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = split_classification(X, y)
print("Train size:", len(y_train), "| Test size:", len(y_test))
cv = make_cv_classification()

**Visibility: new feature names and re-integration.** OneHotEncoder turns one categorical column into several **binary** columns (one per category). Those new columns get **re-integrated** into X by replacing the original column: the model then receives a numeric feature matrix (0/1 instead of text).

In [ ]:
# New feature names (one binary column per category)
enc_feature_names = demo_enc.get_feature_names_out([demo_col])
print(f"{demo_col} → {len(enc_feature_names)} new columns:", enc_feature_names.tolist())

# Build DataFrame of the new columns and show them
df_encoded = pd.DataFrame(demo_encoded, columns=enc_feature_names, index=X_train.index)
print("\nOne-hot encoded columns (first 3 rows):")
display(df_encoded.head(3))

# Re-integration: replace original column with new columns in X (model gets numeric input)
X_train_demo = X_train.drop(columns=[demo_col]).join(df_encoded)
print("\nRe-integrated X (original column removed, new columns added). Shape:", X_train_demo.shape)
display(X_train_demo[enc_feature_names.tolist()].head(3))

In [ ]:
# Make one-hot encoding visible: new feature names and re-integration into X
enc_feature_names = demo_enc.get_feature_names_out([demo_col])
print(f"{demo_col} → {len(enc_feature_names)} new columns:", enc_feature_names.tolist())

df_encoded = pd.DataFrame(demo_encoded, columns=enc_feature_names, index=X_train.index)
print("\nOne-hot encoded columns (first 3 rows):")
display(df_encoded.head(3))

# Re-integration: drop original column, add the new binary columns (so model gets numeric X)
X_train_demo = X_train.drop(columns=[demo_col]).join(df_encoded)
print("\nRe-integrated X: original column removed, new columns added. Shape:", X_train_demo.shape)
display(X_train_demo[enc_feature_names.tolist()].head(3))
print("\nThe full preprocessor (ColumnTransformer) does this for ALL categoricals and concatenates with scaled numerics into one matrix for the model.")

## 5. Preprocessing

As discussed in lecture, many models need numerical values/representation of features to compute the weights.

One of the options to handle categorical features is by using **One-Hot-Encoding**. 

### Why one-hot encoding for categorical columns?

Many algorithms (e.g. Logistic Regression) expect **numeric** input. 

Often, **categorical columns**, e.g.: 
- `Contract`: "Month-to-month", "One year", "Two year"; or 
- `Region`: "North", "South", "East", "West",

have **no meaningful order**. Thus, we cannot assign numbers 1,2,3,4 without implying an order that does not exist. 

**One-hot encoding** turns each category into a separate binary column (1 if that category, 0 otherwise), so the model can use them without imposing an order. 

Below we show which columns are categorical and demonstrate one-hot encoding on one column; then we build the full preprocessor that applies it to all categoricals.

In [ ]:
# Which columns are categorical vs numeric? (We need one-hot encoding for categoricals.)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()
print("Categorical (one-hot encoded):", cat_cols)
print("Numeric (scaled):", num_cols)

# Demo: one-hot encode one column (e.g. Contract) to see the transformation
demo_col = "Contract" if "Contract" in X_train.columns else cat_cols[0]
demo_enc = OneHotEncoder(sparse_output=False)
demo_encoded = demo_enc.fit_transform(X_train[[demo_col]])
print(f"\n{demo_col} categories:", demo_enc.categories_)
print("One-hot encoded shape (n_samples, n_categories):", demo_encoded.shape)
print("First 3 rows encoded:\n", demo_encoded[:3])

In [ ]:
# Full preprocessor: one-hot encode all categoricals + scale numerics (same logic as the demo above).
preprocessor = make_classification_preprocessor(X_train)
# We will use Pipeline(preprocessor, model) so that CV fits preprocessor per fold (no leakage)

## 6. Baseline: majority class predictor

Before any complex model, we evaluate a **baseline** (classification: predict the majority class). All other models must be compared against this using the same protocol.

*This section uses the **baseline** model (`DummyClassifier`), not Logistic Regression. Section 7 trains and evaluates **Logistic Regression**.*

In [ ]:
# Baseline: majority class (stratified CV on training set only)
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
])
baseline_cv_metrics = evaluate_classification_cv(baseline_pipeline, X_train, y_train, cv)
print("Baseline CV metrics (on train):", baseline_cv_metrics)

*Note:* The baseline CV accuracy (on the training set) and baseline test accuracy (on the held-out test set) can differ slightly (e.g. 0.7342 vs 0.7341) because train and test are different samples — the majority-class rate is not identical in both. This is expected and reproducible for a fixed `RANDOM_STATE`. Precision/recall/F1 are 0 because the baseline never predicts "Churn".

In [ ]:
# Retrain baseline on full training set; evaluate exactly once on test set
baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)
baseline_test_metrics = cls_metrics(y_test, y_pred_baseline)
print("Baseline test metrics (final, one-time evaluation):", baseline_test_metrics)

## 7. Model comparison (CV on training set only)

**Define model candidates first, then run CV.**  
**Theoretical path:** Before cross-validation, we **pre-define** all model candidates (structure, hyperparameters, or architecture). Then we run each through the same CV and compare. For complex models (e.g. neural nets with layers and activations, or custom estimators), you must build the model object first; only then can you pass it to CV.

**Code / package practice:** For simple models like Logistic Regression, the library allows a one-liner (e.g. instantiating inside the CV call). So in code you *could* define the model inline. Here we **pre-define** the second candidate (Logistic Regression) before the CV cell anyway: it keeps the workflow clear (candidates → then CV), matches the theoretical path, and scales to cases where instantiation is a separate step.

In [ ]:
# Candidate 1: Baseline (already defined in Section 6 as baseline_pipeline)
# Candidate 2: Logistic Regression — instantiate here before CV
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

**Before the comparison CV — your turn (case study):**  
Defining models, hyperparameter spaces, and tuning belongs **before** you run the comparison CV below. Add **at least 2 more model candidates** (e.g. Random Forest, Gradient Boosting, k-NN, or others). For each new model:

1. **Define the model** (instantiate with default or chosen hyperparameters) in a new cell above, like we did for Logistic Regression.
2. **Define a hyperparameter search space** (e.g. a dict of parameter names and lists of values, or a distribution for randomized search).
3. **Run hyperparameter tuning** on the **training set only** (e.g. `GridSearchCV` or `RandomizedSearchCV` with the same `cv` and a chosen scoring metric). Tuning uses CV internally to select the best parameters.
4. Include the **best-tuned pipelines** (or the best estimator from each search) in the **comparison CV** in the next cell, so you run all candidates (baseline, LR, and your tuned models) through the same CV and compare.

Once you have all candidates defined (and tuned), the **next step** is the comparison CV.

**Comparison CV:** Run all candidates (baseline, Logistic Regression, and any you added above) through the **same CV** on the training set and compare metrics to select the best. Hyperparameter tuning (if done) adds more candidates and also uses only the training set (e.g. GridSearchCV on train).

In [ ]:
# Run all pre-defined candidates through the same CV
baseline_cv_metrics = evaluate_classification_cv(baseline_pipeline, X_train, y_train, cv)
lr_cv_metrics = evaluate_classification_cv(lr_pipeline, X_train, y_train, cv)

# Compare CV outcomes for all candidates
cv_comparison = pd.DataFrame({
    "Baseline (majority class)": baseline_cv_metrics,
    "Logistic Regression": lr_cv_metrics,
}).T
print("CV metrics (on train) — model candidates compared:")
display(cv_comparison)

**Challenge: Fixed train–validation split vs cross-validation**

Run the cell below. It takes the same training set and splits it **once** into a train part and a validation part, fits Logistic Regression on the train part only, and evaluates on that single validation part. **Compare the same model (Logistic Regression):** Use the **CV metrics** from Section 7 above (LR, average over 5 folds). Do **not** compare with Section 6 — that is the **baseline**, not LR. Here we compare **LR with CV** vs **LR with one split**.

**Reflect:**
- Is the fixed-split result also "correct"? Why might the numbers differ (e.g. accuracy, or precision/recall/F1)?
- If on the fixed split you see a different metrics values, e.g. **precision = 0, recall = 0, F1 = 0**, but CV gives non-zero values: what does that mean for the model’s predictions on that one validation set? What is the reason?
- What could you do in practice? (e.g. inspect predictions, use a stratified split, prefer CV for model comparison, or adjust the decision threshold.)

## 8. Retrain selected model and final test evaluation

We choose the best configuration based on CV (here: Logistic Regression). We **retrain** on the **full training set** and evaluate **exactly once** on the test set.

In [ ]:
# Retrain on full training set
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]  # probability of positive class (Churn)
lr_test_metrics = cls_metrics(y_test, y_pred_lr, y_proba=y_proba_lr)
print("Logistic Regression test metrics (final, one-time evaluation):", lr_test_metrics)

In [ ]:
# Compare baseline vs Logistic Regression on test set
pd.DataFrame({
    "Baseline": baseline_test_metrics,
    "Logistic Regression": lr_test_metrics,
}).T

## 9. Metric justification

With **class imbalance** (e.g. fewer churners than non-churners):

- **Accuracy** can be misleading: predicting "No churn" for everyone can yield high accuracy but is useless for identifying churners.
- **Precision** (among predicted churners, how many actually churned) matters for targeting interventions.
- **Recall** (among actual churners, how many we found) matters when missing a churner is costly.
- **F1** balances precision and recall (harmonic mean).
- **ROC-AUC** summarizes ranking performance across thresholds and is robust to imbalance.

**Justify your choice** depending on the business goal (e.g. "We optimize for recall because missing churners is costlier than false alarms").

## 10. Error analysis

Inspect where the model fails: e.g. confusion matrix, and optionally errors by segment (e.g. contract type, tenure). Use the **test set** only for this descriptive analysis after the single final evaluation.


In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_baseline, ax=ax[0])
ax[0].set_title("Baseline")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=ax[1])
ax[1].set_title("Logistic Regression")
plt.tight_layout()
plt.show()

---

**Next steps (for your case study):** Add more models (e.g. tree-based), tune hyperparameters via cross-validation on the training set only, deepen error analysis (e.g. by contract type or tenure), and justify your final metric and model choice in writing.